# Vortex Fitting and Detection Notebook

This notebook runs the vortex fitting script on GLORYS data using the 'thesis' kernel.

In [ ]:
import sys
import os
sys.path.append('SOFTX-D-20-00015-master')

import argparse
from vortexfitting import fitting
from vortexfitting import schemes
from vortexfitting import detection
from vortexfitting import output
from vortexfitting import classes

In [ ]:
import matplotlib
matplotlib.use('Agg')  # For headless plotting

In [ ]:
# Set Parameters
input_filename = 'SOFTX-D-20-00015-master/data/GPGP_oct2020_45N20S-130E-155W.nc'
output_directory = 'results'
scheme = 22
detection_method = 'swirling'
detection_threshold = 0.0
box_size = 6
flip_axis = False
mean_filename = '/'
plot_method = 'fit'
xy_location = [0, 0]
first = 0
last = 0
step = 1
rmax = 8000
file_type = 'glorys'
correlation_threshold = 0.6
output_format = 'png'

In [ ]:
# Load Data
if last < first:
    last = first

for time_step in range(first, last + 1, step):
    if not os.path.exists(input_filename.format(time_step)):
        print('The input file does not exist. Exiting.')
        sys.exit()

    print('\nOpening file: ', input_filename.format(time_step), ', file type: ', file_type)
    if mean_filename != '/':
        print('Opening mean field: ', mean_filename)

    vfield = classes.VelocityField(input_filename, time_step, mean_filename, file_type)


Opening file:  SOFTX-D-20-00015-master/data/GPGP_oct2020_45N20S-130E-155W.nc , file type:  dns


KeyError: 'velocity_x'

In [ ]:
# Compute Derivatives
if scheme == 4:
    vfield.derivative = schemes.fourth_order_diff(vfield)
elif scheme == 2:
    vfield.derivative = schemes.second_order_diff(vfield)
elif scheme == 22:
    vfield.derivative = schemes.least_square_diff(vfield)
else:
    print('No scheme', scheme, 'found. Exiting!')
    sys.exit()

Difference scheme: least-square filter


In [ ]:
# Compute Vorticity
vorticity = vfield.derivative['dvdx'] - vfield.derivative['dudy']

In [ ]:
# Detect Vortices
detection_field = []
if detection_method == 'Q':
    detection_field = detection.calc_q_criterion(vfield)
elif detection_method == 'swirling':
    detection_field = detection.calc_swirling(vfield)
elif detection_method == 'delta':
    detection_field = detection.calc_delta_criterion(vfield)

if vfield.normalization_flag:
    print('Normalization for ', vfield.normalization_direction, ' direction')
    detection_field = fitting.normalize(detection_field, vfield.normalization_direction)

Detection method: 2D swirling strength
Max value of swirling:  0.0


In [ ]:
# Find Peaks
print('Threshold=', detection_threshold, ', box size=', box_size)
peaks = fitting.find_peaks(detection_field, detection_threshold, box_size)
print('Vortices found: ', len(peaks[0]))

Threshold= 0.0 , box size= 6
Vortices found:  1069


In [ ]:
# Determine Rotation Direction
vortices_counterclockwise, vortices_clockwise = fitting.direction_rotation(vorticity, peaks)

In [ ]:
# Fit Vortices
vortices = list()
if (plot_method == 'fit') and (xy_location == [0, 0]):
    vortices = fitting.get_vortices(vfield, peaks, vorticity, rmax, correlation_threshold)
    print('---- Accepted vortices ----')
    print(len(vortices))
else:
    print('No fitting')

0 Processing detected swirling at (x, y) 3 2


ValueError: operands could not be broadcast together with shapes (0,) (4,4) 

In [ ]:
# Plot Results
if xy_location != [0, 0]:
    x_location = int(xy_location[0])
    y_location = int(xy_location[1])
    detection_field_window = detection_field[y_location - 10:y_location + 10, x_location - 10:x_location + 10]
    x_index, y_index, u_data, v_data = fitting.window(vfield, x_location, y_location, 10)
    fitting.plot_quiver(x_index, y_index, u_data, v_data, detection_field_window)
if plot_method == 'detect':
    fitting.plot_detect(vortices_counterclockwise, vortices_clockwise, detection_field, flip_axis)
if plot_method == 'fields':
    fitting.plot_fields(vfield, vorticity)
if plot_method == 'fit':
    output.create(output_directory, None)  # args not needed
    fitting.plot_accepted(vfield, vortices, detection_field, output_directory, time_step, output_format)
    fitting.plot_vortex(vfield, vortices, output_directory, time_step, output_format)
    output.write(vortices, output_directory, time_step)